# テーマT Phase T1：符号地図サーベイ v0.3（統一・公開版）
E7〜E10（クライン空間族）のパラメータ点ごとに，鏡映統計S⁺/S⁻の符号を地図化します。

**v0.1からの設計変更（変更履歴として公開）**
1. **不変面の自動導出**：各多様体のホロノミー行列 M をソースから自動抽出し，
   反転型（det=−1, 固有値+1,+1,−1）の面法線・半回転型（+1,−1,−1）の軸を
   生成元とその2次積まで機械的に列挙。v0.1は全多様体をŷ軸のみで評価しており，
   E8/E10の第2の不変面（x̂面等）を見落としていた——この種の誤りを構造的に排除。
2. **等方基準の物理化**：単位行列（ℓ平坦重み）→ CAMB C_ℓ対角（物理重み）。
3. **検証バッテリー**：C_ℓ形状照合・エルミート性・**E7系のv0.1数値との一致検証**
   （固定シードのため±0.02で一致するはず＝新旧プログラムの相互検証）。
4. 大域走査（全方向のminS⁺/minS⁻）を全点で記録（どの面の異常も取りこぼさない）。

既存の共分散（Drive上runs/）は自動再利用。新規点のみ計算（約5分/点）。

**v0.3**：v0.2のC_ℓ形状検査は等方極限用の±15%を全点に誤適用しており，
コンパクト位相が離散kスペクトルでℓ別平均パワーを変調する正しい物理
（rel〜0.9-1.5）を警告として誤検知していた。v0.3では (i) 全点検査は
「物理重みの健全性帯 [0.5, 2.0]」（単位対角規格化の誤り rel≈3.3 は検出可能）に
修正し，位相由来の変調は警告でなく情報としてCSVへ記録，(ii) 厳密±15%検査は
等方極限（全辺1.4）専用のTier Bセルとして実施。

In [ ]:
# ---- 設定 ----
TIME_BUDGET_MIN=110
import os, sys, subprocess, time, json, glob, re
IN_COLAB=os.path.isdir('/content')
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    if not os.path.isdir('/content/drive/MyDrive'):
        raise RuntimeError('★Driveマウント失敗：再実行してください')
    BASE='/content/drive/MyDrive/mirror_topology'
else: BASE='mirror_topology_out'
os.makedirs(BASE,exist_ok=True)
for p in ['healpy','camb','numba','quaternionic','spherical','tqdm']:
    subprocess.run([sys.executable,'-m','pip','install','-q',p])
import numpy as np, pandas as pd, healpy as hp
if not os.path.isdir(os.path.join(BASE,'CMBtopology')):
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/CompactCollaboration/CMBtopology.git',
        os.path.join(BASE,'CMBtopology')])
os.chdir(os.path.join(BASE,'CMBtopology')); sys.path.insert(0,'.')
DEADLINE=time.time()+TIME_BUDGET_MIN*60
def left(): return DEADLINE-time.time()
LMAX=4
print('BASE =',BASE)

In [ ]:
# ---- 検証済みモジュール ----
open('phase2_core.py','w').write(r'''# -*- coding: utf-8 -*-
"""Phase 2 較正基盤コア（v0.1）
- 処理構成マニフェスト（計画書v1.0 §3準拠）
- CRNマスター実現（synalm, seed 0..999, lmax=128）
- 統計①②③④の null 分布計算（チェックポイント/再開対応）
"""
import os, json, time
import numpy as np, healpy as hp

# ---------- 基本設定 ----------
LMAX_MASTER = 128
NSIM_FULL = 1000
FID_CL_FILE = 'CMBanom/data/real/COM_PowerSpect_CMB-base-plikHM-TTTEEE-lowl-lowE-lensing-minimum-theory_R3.01.txt'
COMMON_MASK_128 = 'CMBanom/data/masks/com_mask_cutoff_0.9_nside_128.fits'

def load_fid_cl(lmax=LMAX_MASTER):
    dat = np.loadtxt(FID_CL_FILE, skiprows=1)
    ll = np.arange(lmax + 1); cl = np.zeros(lmax + 1)
    n = min(lmax - 1, dat.shape[0])
    cl[2:2 + n] = dat[:n, 1] * 2 * np.pi / (ll[2:2 + n] * (ll[2:2 + n] + 1))
    return cl

# ---------- 伝達関数・マスク ----------
PIXWIN_CACHE = 'pixwin_cache'
def pixwin_pad(nside, lmax):
    fn = os.path.join(PIXWIN_CACHE, f'pixel_window_n{nside:04d}.fits')
    if os.path.exists(fn):
        from astropy.io import fits as _f
        with _f.open(fn) as h:
            pw = np.asarray(h[1].data['TEMPERATURE']).ravel()
    else:
        pw = hp.pixwin(nside)   # Colabでは通常経路（初回のみDL）
    return np.pad(pw, (0, max(0, lmax + 1 - len(pw))), mode='edge')[:lmax + 1]

def transfer(nside, smooth, lmax=LMAX_MASTER):
    """構成の伝達関数 b_ℓ p_ℓ。smooth ∈ {'planck','none','fix5deg'}"""
    pw = pixwin_pad(nside, lmax)
    if smooth == 'planck':
        fwhm_arcmin = 640.0 * 16.0 / nside
        return hp.gauss_beam(np.radians(fwhm_arcmin / 60.), lmax=lmax) * pw
    if smooth == 'fix5deg':
        return hp.gauss_beam(np.radians(5.0), lmax=lmax) * pw
    if smooth == 'none':
        return pw.copy()
    raise ValueError(smooth)

def dilate_mask(bad, nside, deg):
    """マスク（bad=True）を約deg度拡張（近傍膨張の反復）"""
    pixsize_deg = np.degrees(hp.nside2resol(nside))
    n_iter = max(1, int(np.ceil(deg / pixsize_deg)))
    bad = bad.copy()
    for _ in range(n_iter):
        idx = np.where(bad)[0]
        nb = hp.get_all_neighbours(nside, idx)
        bad[nb[nb >= 0]] = True
    return bad

def _dilate_n(bad, nside, n_iter):
    bad = bad.copy()
    for _ in range(n_iter):
        idx = np.where(bad)[0]
        nb = hp.get_all_neighbours(nside, idx)
        bad[nb[nb >= 0]] = True
    return bad

def _erode_n(bad, nside, n_iter):
    return ~_dilate_n(~bad, nside, n_iter)

def make_mask(nside, level):
    """level ∈ {'full','common','ext'} → bool（True=使用画素）
    ext = Planck 2015 XVI Table 12方式：共通マスクの拡散（銀河）成分のみを5°拡張し
          点源穴は拡張しない（補助マスク規定）。128で構成してから縮退。"""
    npix = hp.nside2npix(nside)
    if level == 'full':
        return np.ones(npix, bool)
    m128 = hp.read_map(COMMON_MASK_128)
    if level == 'common':
        return hp.ud_grade(m128, nside) >= 0.9
    if level == 'ext':
        bad128 = m128 < 0.5
        pix_deg = np.degrees(hp.nside2resol(128))          # ≈0.46°
        n_open = 2                                          # 開演算で点源(≲1°)を除去
        diffuse = _dilate_n(_erode_n(bad128, 128, n_open), 128, n_open)
        n5 = int(np.ceil(5.0 / pix_deg))                    # 5°膨張
        bad_ext = bad128 | _dilate_n(diffuse, 128, n5)
        return hp.ud_grade((~bad_ext).astype(float), nside) >= 0.9
    raise ValueError(level)

# ---------- CRNマスター実現 ----------
def gen_master_alm(outfile, nsim=NSIM_FULL, lmax=LMAX_MASTER):
    if os.path.exists(outfile):
        return np.load(outfile)['alms']
    cl = load_fid_cl(lmax)
    alms = np.empty((nsim, hp.Alm.getsize(lmax)), dtype=np.complex128)
    for s in range(nsim):
        np.random.seed(s)
        alms[s] = hp.synalm(cl, lmax=lmax)
    np.savez_compressed(outfile, alms=alms, lmax=lmax, nsim=nsim,
                        note='CRN master: fiducial PR3 bestfit, seeds 0..nsim-1')
    return alms

def config_maps(alms, nside, smooth, route='harmonic', lmax=LMAX_MASTER):
    """マスター実現→構成マップ群 (nsim, npix)"""
    bl = transfer(nside, smooth, lmax)
    npix = hp.nside2npix(nside)
    out = np.empty((alms.shape[0], npix))
    if route == 'harmonic':
        for s in range(alms.shape[0]):
            out[s] = hp.alm2map(hp.almxfl(alms[s], bl), nside)
    elif route == 'udgrade':   # 高解像度で実体化→画素平均（quadrature軸）
        nhi = min(4 * nside, 64)
        bl_hi = transfer(nside, smooth, lmax) / pixwin_pad(nside, lmax) * pixwin_pad(nhi, lmax)
        for s in range(alms.shape[0]):
            out[s] = hp.ud_grade(hp.alm2map(hp.almxfl(alms[s], bl_hi), nhi), nside)
    else:
        raise ValueError(route)
    return out

# ---------- 統計②：鏡映パリティ（マスク対応） ----------
class MirrorStat:
    def __init__(self, nside, mask):
        npix = hp.nside2npix(nside)
        vecs = np.array(hp.pix2vec(nside, np.arange(npix))).T
        R = np.empty((npix, npix), dtype=np.int32)
        for i in range(npix):
            n = vecs[i]
            refl = vecs - 2.0 * np.outer(vecs @ n, n)
            R[i] = hp.vec2pix(nside, refl[:, 0], refl[:, 1], refl[:, 2])
        self.R = R
        self.valid = mask[None, :] & mask[R]          # 双方が有効な対のみ
        self.cnt = np.maximum(self.valid.sum(axis=1), 1)
        self.mask = mask

    def min_S(self, mp, chunk=1024, mondip=True):
        if mondip:
            mp = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, mp, hp.UNSEEN))))
        mp = np.where(self.mask, mp, 0.0).astype(np.float32)
        nd = self.R.shape[0]
        Sp = np.empty(nd, np.float32); Sm = np.empty(nd, np.float32)
        for a in range(0, nd, chunk):
            Tr = mp[self.R[a:a + chunk]]
            V = self.valid[a:a + chunk]
            Sp[a:a + chunk] = np.sum(V * (0.5 * (mp[None, :] + Tr)) ** 2, axis=1) / self.cnt[a:a + chunk]
            Sm[a:a + chunk] = np.sum(V * (0.5 * (mp[None, :] - Tr)) ** 2, axis=1) / self.cnt[a:a + chunk]
        return float(Sp.min()), float(Sm.min())

# ---------- 統計①：R/D（QML / MASTER / naive fsky） ----------
def C_pm(cl_from2, lmax):
    ell = np.arange(2, lmax + 1)
    Dl = ell * (ell + 1.) / (2 * np.pi) * cl_from2[:lmax - 1]
    ev = (ell % 2 == 0)
    return Dl[ev].sum() / (lmax - 1), Dl[~ev].sum() / (lmax - 1)

def RD_traj(cl_from2, lmaxes):
    out = np.empty((len(lmaxes), 2))
    for i, L in enumerate(lmaxes):
        p, m = C_pm(cl_from2, L)
        out[i] = (p / m, p - m)
    return out

def lmax_est_of(nside):
    return int(min(40, 2.5 * nside))

# ---------- 統計③：多重極ベクトル（polyMV非依存・性質テスト済） ----------
from scipy.special import gammaln as _gln

def multipole_vectors(alm, lmax, ell):
    a = np.zeros(2*ell+1, dtype=complex)
    for m in range(0, ell+1):
        v = alm[hp.Alm.getidx(lmax, ell, m)]
        a[ell+m] = v
        if m: a[ell-m] = (-1)**m * np.conj(v)
    k = np.arange(2*ell+1)
    logC = _gln(2*ell+1) - _gln(k+1) - _gln(2*ell-k+1)
    c = np.sqrt(np.exp(logC)) * a
    roots = np.roots(c[::-1])
    th = 2*np.arctan(np.abs(roots)); ph = np.angle(roots) + np.pi
    v = np.stack([np.sin(th)*np.cos(ph), np.sin(th)*np.sin(ph), np.cos(th)], 1)
    v = np.where(v[:, 2:3] >= 0, v, -v)
    keep = []
    for i in range(len(v)):
        if not any(np.dot(v[i], v[j]) > 0.999 for j in keep): keep.append(i)
    return v[keep[:ell]]

def stat3_SQO(maps, mask, lmax_alm=8):
    """素朴カットスカイalm経路（第1弾で検証済みの規約）でS_QO"""
    vals = np.empty(maps.shape[0])
    fmask = mask.astype(float)
    for s in range(maps.shape[0]):
        alm = hp.map2alm(maps[s]*fmask, lmax=lmax_alm, iter=3)
        v2 = multipole_vectors(alm, lmax_alm, 2)
        v3 = multipole_vectors(alm, lmax_alm, 3)
        w2 = np.cross(v2[0], v2[1])
        w3 = [np.cross(v3[i], v3[j]) for i in range(3) for j in range(i+1, 3)]
        vals[s] = np.mean([abs(np.dot(w2, w)) for w in w3])
    return vals

# ---------- 実行系（チェックポイント） ----------
def null_dir(base, cfg_id):
    d = os.path.join(base, cfg_id); os.makedirs(d, exist_ok=True); return d

def true_varlvmap(lvmaps, lvmask, mean_lvmap):
    """正しい逆分散重み用の規格化分散（リポジトリ版get_varlvmapは定数(N-1)²/Nになるバグ）"""
    return np.where(lvmask == 1.,
                    np.mean((lvmaps - mean_lvmap) ** 2, axis=0) / np.maximum(mean_lvmap, 1e-30) ** 2,
                    1.)

def run_stat2(base, cfg_id, maps, nside, mask, chunk_ckpt=100, mondip=True):
    """②のnull分布（部分保存・再開対応）"""
    d = null_dir(base, cfg_id); fn = os.path.join(d, 'stat2.npz')
    done = 0; minSp = []; minSm = []
    if os.path.exists(fn):
        z = np.load(fn)
        if z['complete']: return
        minSp, minSm, done = list(z['minSp']), list(z['minSm']), int(z['done'])
    ms = MirrorStat(nside, mask)
    for s in range(done, maps.shape[0]):
        sp, sm = ms.min_S(maps[s], mondip=mondip)
        minSp.append(sp); minSm.append(sm)
        if (s + 1) % chunk_ckpt == 0 or s == maps.shape[0] - 1:
            np.savez(fn, minSp=minSp, minSm=minSm, done=s + 1,
                     complete=(s == maps.shape[0] - 1))
    return np.array(minSp), np.array(minSm)
''')
open('plane_mirror.py','w').write(r'''# -*- coding: utf-8 -*-
"""plane_mirror.py — テーマA: 鏡映反対称性の高速バッチ評価
MirrorStat（phase2_core）と厳密同一の定義で，方向走査をマップ束一括化。
検証済み（2026-08-15）: MirrorStatとの max相対差 1.2e-6（float32水準）。
計時: N16 33ms/マップ（10^5=55分）, N32 0.9s/マップ（10^4=2.5時間）。
"""
import numpy as np
import healpy as hp


class MirrorBatch:
    def __init__(self, ms):
        self.R, self.valid, self.cnt, self.mask = ms.R, ms.valid, ms.cnt, ms.mask

    def min_S_batch(self, maps, mondip=True, return_argmin=False):
        B = maps.shape[0]
        T = np.empty((maps.shape[1], B), np.float32)
        for b in range(B):
            m = maps[b]
            if mondip:
                m = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, m, hp.UNSEEN))))
            T[:, b] = np.where(self.mask, m, 0.0)
        nd = self.R.shape[0]
        minSp = np.full(B, np.inf, np.float32); minSm = np.full(B, np.inf, np.float32)
        argp = np.zeros(B, np.int32); argm = np.zeros(B, np.int32)
        for d in range(nd):
            v = self.valid[d]
            if not v.any():
                continue
            Tr = T[self.R[d]]
            Tv, Trv = T[v], Tr[v]
            Sp = np.einsum('ib,ib->b', 0.5 * (Tv + Trv), 0.5 * (Tv + Trv)) / self.cnt[d]
            Sm = np.einsum('ib,ib->b', 0.5 * (Tv - Trv), 0.5 * (Tv - Trv)) / self.cnt[d]
            better = Sp < minSp
            if return_argmin:
                argp = np.where(better, d, argp)
            minSp = np.where(better, Sp, minSp)
            better = Sm < minSm
            if return_argmin:
                argm = np.where(better, d, argm)
            minSm = np.where(better, Sm, minSm)
        if return_argmin:
            return minSp, minSm, argp, argm
        return minSp, minSm


def axis_lb(nside, d):
    """方向画素番号 → 銀経緯 (l, b) [deg]"""
    th, ph = hp.pix2ang(nside, int(d))
    return float(np.degrees(ph)), float(90.0 - np.degrees(th))


class FixedAxisMirror:
    """凍結軸 d* での対称成分解析（O(npix)/マップ）"""
    def __init__(self, ms, d_star):
        self.r = ms.R[d_star]
        self.v = ms.valid[d_star]
        self.cnt = ms.cnt[d_star]
        self.mask = ms.mask

    def S_plus(self, mp, mondip=True):
        if mondip:
            mp = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, mp, hp.UNSEEN))))
        T = np.where(self.mask, mp, 0.0)
        s = 0.5 * (T + T[self.r])
        return float(np.sum(self.v * s * s) / self.cnt)

    def S_plus_batch(self, maps, mondip=True):
        return np.array([self.S_plus(maps[b], mondip) for b in range(maps.shape[0])])

    def band_decompose(self, alm128, nside, transfer_fl, bands, mondip=True):
        """ソースalm(lmax128)を帯域分解し，帯域別S⁺と全帯域和・交差項を返す"""
        import numpy as _np
        LMAX = hp.Alm.getlmax(len(alm128))
        ell = _np.arange(LMAX + 1)
        out = {}
        s_parts = []
        for (l0, l1) in bands:
            w = ((ell >= l0) & (ell <= l1)).astype(float) * transfer_fl
            mb = hp.alm2map(hp.almxfl(alm128.copy(), w), nside)
            if mondip and l0 <= 1:
                pass
            T = _np.where(self.mask, mb, 0.0)
            if mondip:
                T = _np.where(self.mask,
                              _np.asarray(hp.remove_dipole(hp.ma(_np.where(self.mask, mb, hp.UNSEEN)))), 0.0)
            s = 0.5 * (T + T[self.r])
            s_parts.append(s)
            out[f'S{l0}_{l1}'] = float(_np.sum(self.v * s * s) / self.cnt)
        stot = _np.sum(s_parts, axis=0)
        out['S_sum_bands'] = float(_np.sum(self.v * stot * stot) / self.cnt)
        return out

    def exclusion_scan(self, mp, nside_scan=8, radius_deg=15.0, mondip=True):
        """半径radius_degの円盤を各走査位置で（鏡映相手も対称に）除外した際の
        ln S⁺ の変化 Δ(q) を返す（正＝除外で対称性が増える＝その領域が反対称の担い手）"""
        if mondip:
            mp = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, mp, hp.UNSEEN))))
        T = np.where(self.mask, mp, 0.0)
        s = 0.5 * (T + T[self.r])
        base_num = np.sum(self.v * s * s)
        base = base_num / self.cnt
        npix_scan = hp.nside2npix(nside_scan)
        nside_map = hp.npix2nside(len(T))
        delta = np.zeros(npix_scan)
        for q in range(npix_scan):
            vec = hp.pix2vec(nside_scan, q)
            disc = hp.query_disc(nside_map, vec, np.radians(radius_deg))
            ex = np.zeros(len(T), bool); ex[disc] = True
            ex = ex | ex[self.r]                       # 対称除外
            v2 = self.v & ~ex
            c2 = max(v2.sum(), 1)
            S2 = np.sum(v2 * s * s) / c2
            delta[q] = np.log(S2 / base) if S2 > 0 else 0.0
        return delta, base


def with_mask(ms, mask):
    """R表を共有して別マスクのMirrorStat相当を作る（N32のR再構築20秒を節約）"""
    obj = type(ms).__new__(type(ms))
    obj.R = ms.R
    obj.mask = mask
    obj.valid = mask[None, :] & mask[ms.R]
    obj.cnt = np.maximum(obj.valid.sum(axis=1), 1)
    return obj


def scan_S(ms, mp, mondip=True):
    """1マップの全方向S±(n̂)地形を返す（軸の縮退・地形幅の解析用）"""
    if mondip:
        mp = np.asarray(hp.remove_dipole(hp.ma(np.where(ms.mask, mp, hp.UNSEEN))))
    T = np.where(ms.mask, mp, 0.0).astype(np.float32)
    nd = ms.R.shape[0]
    Sp = np.empty(nd, np.float32); Sm = np.empty(nd, np.float32)
    for d in range(nd):
        v = ms.valid[d]
        Tr = T[ms.R[d]]
        Sp[d] = np.sum(v * (0.5 * (T + Tr)) ** 2) / ms.cnt[d]
        Sm[d] = np.sum(v * (0.5 * (T - Tr)) ** 2) / ms.cnt[d]
    return Sp, Sm


def axis_sep_deg(nside, d1, d2):
    """2軸の分離角[deg]（鏡映面法線は±同一視 → 90°超は補角）"""
    v1 = np.array(hp.pix2vec(nside, int(d1)))
    v2 = np.array(hp.pix2vec(nside, int(d2)))
    ang = np.degrees(np.arccos(np.clip(abs(v1 @ v2), -1, 1)))
    return float(ang)
''')
open('topo_bridge.py','w').write(r'''# -*- coding: utf-8 -*-
"""topo_bridge.py — CMBtopology共分散 ↔ 凍結鏡映統計の配線（テーマT）
検証: (l,m)順序=ℓ2..lmax×m=-ℓ..ℓ / m<0→healpy変換 / T0で実現経路の動作実証済み"""
import numpy as np
import healpy as hp


def load_cov(run_dir, lmax):
    import glob, os
    f = glob.glob(os.path.join(run_dir, f'TT_corr_matrix_l_2_{lmax}_lp_2_{lmax}.npy'))
    assert f, f'covariance not found in {run_dir}'
    M = np.load(f[0])
    lm = [(l, m) for l in range(2, lmax+1) for m in range(-l, l+1)]
    assert M.shape == (len(lm), len(lm)), (M.shape, len(lm))
    assert np.allclose(M, M.conj().T, atol=1e-8 * np.abs(M).max())
    return M, lm


def sample_alms(M, lm, nreal, seed0=0, jitter=1e-9):
    lmax = max(l for l, _ in lm)
    A = np.linalg.cholesky(M + jitter * np.abs(np.diag(M)).max() * np.eye(len(lm)))
    out = np.zeros((nreal, hp.Alm.getsize(lmax)), complex)
    for r in range(nreal):
        rng = np.random.default_rng(seed0 + r)
        z = (rng.normal(size=len(lm)) + 1j * rng.normal(size=len(lm))) / np.sqrt(2)
        v = A @ z
        for i, (l, m) in enumerate(lm):
            if m >= 0:
                out[r, hp.Alm.getidx(lmax, l, m)] += v[i] / np.sqrt(2 if m > 0 else 1)
            else:
                out[r, hp.Alm.getidx(lmax, l, -m)] += ((-1) ** m) * np.conj(v[i]) / np.sqrt(2)
    return out


def alms_to_maps(alms, nside):
    return np.array([hp.alm2map(np.ascontiguousarray(a), nside) for a in alms], np.float32)


def mirror_report(maps, ms, mb, fa_y, nscan=60):
    """S⁺/S⁻の大域最小・固定ŷ軸値・軸集中をまとめて返す"""
    import plane_mirror as pm
    Sp, Sm, ap, am = mb.min_S_batch(maps, mondip=False, return_argmin=True)
    Sy = np.array([fa_y.S_plus(maps[r], mondip=False) for r in range(maps.shape[0])])
    med = np.array([np.median(pm.scan_S(ms, maps[r], mondip=False)[0])
                    for r in range(min(nscan, maps.shape[0]))])
    from collections import Counter
    occ = sum(c for _, c in Counter(ap.tolist()).most_common(3)) / maps.shape[0]
    return dict(minSp_ratio=float(np.median(Sp[:len(med)] / med)),
                minSm_ratio=float(np.median(Sm[:len(med)] / med)),
                Sy_ratio=float(np.median(Sy[:len(med)] / med)),
                axis_occ3=float(occ))
''')
import types; sys.modules.setdefault('polymv', types.ModuleType('polymv'))
import importlib, phase2_core as p2, plane_mirror as pm, topo_bridge as tb
for m in (p2,pm,tb): importlib.reload(m)
print('モジュールOK')

In [ ]:
# ---- 不変面の自動導出（ホロノミー行列をソースから抽出） ----
def extract_holonomies(top):
    src=open(f'topology/src/{top}.py').read()
    mats={}
    for m in re.finditer(r'M_([A-Z])\s*=\s*np\.(diag|array)\(', src):
        name='M_'+m.group(1); i=m.end()-1; depth=0
        for j in range(i,len(src)):
            if src[j]=='(': depth+=1
            elif src[j]==')':
                depth-=1
                if depth==0: break
        expr='np.'+m.group(2)+src[i:j+1]
        try: mats[name]=np.array(eval(expr,{'np':np}),float)
        except Exception: pass
    return mats
def invariant_axes(top):
    mats=extract_holonomies(top)
    Ms=list(mats.values())
    prods={**mats}
    for a in mats:
        for b in mats:
            prods[f'{a}@{b}']=mats[a]@mats[b]
    axes={}
    for nm,M in prods.items():
        if np.allclose(M,np.eye(3)): continue
        w,v=np.linalg.eigh((M+M.T)/2)
        ev=np.round(np.linalg.eigvals(M)).real
        if sorted(ev.tolist())==[-1,1,1]:      # 反転型: 面法線=−1固有ベクトル
            n=np.linalg.eigh(M)[1][:,np.argmin(np.linalg.eigh(M)[0])]
            axes[f'refl:{nm}']=n/np.linalg.norm(n)
        elif sorted(ev.tolist())==[-1,-1,1]:   # 半回転型: 軸=+1固有ベクトル
            n=np.linalg.eigh(M)[1][:,np.argmax(np.linalg.eigh(M)[0])]
            axes[f'halfturn:{nm}']=n/np.linalg.norm(n)
    # 同一軸（±込み）の重複除去
    uniq={}
    for k,n in axes.items():
        if not any(abs(abs(n@u)-1)<1e-6 for u in uniq.values()): uniq[k]=n
    return uniq
def axkey(name):
    t,g=name.split(':')
    return t+'_'+g.replace('M_','').replace('@','')
AXES={t:invariant_axes(t) for t in ['E7','E8','E9','E10']}
for t,a in AXES.items():
    print(t, {k: np.round(v,2).tolist() for k,v in a.items()})

In [ ]:
# ---- 走査点リストと find-or-run ----
def E7p(top='E7',LAx=1,LAy=1,L1y=1,L2x=1,L2z=1,x0y=0.0,tag=''):
    return dict(topology=top,params=dict(LAx=LAx,LAy=LAy,L1y=L1y,L2x=L2x,L2z=L2z),
                x0=[0.0,x0y,0.0],tag=tag)
def KBp(top,LAx=1,LAy=0,LBx=0,LBz=1,LCy=1,x0y=0.0,tag=''):
    return dict(topology=top,params=dict(LAx=LAx,LAy=LAy,LBx=LBx,LBz=LBz,LCy=LCy),
                x0=[0.0,x0y,0.0],tag=tag)
POINTS=[]
for L1y in [1.0,0.85,0.7,0.6]:
    for frac,ft in [(0.0,'g0'),(0.25,'gq'),(0.5,'gh')]:
        x0y=round((1.0-frac*L1y)/2 % L1y,3)
        POINTS.append(E7p(L1y=L1y,x0y=x0y,tag=f'E7_L1y{L1y}_{ft}'))
POINTS+=[E7p(LAx=0.7,tag='E7_LAx0.7'),E7p(LAx=1.3,tag='E7_LAx1.3'),E7p(L2x=0.5,tag='E7_tilt'),
         KBp('E8',tag='E8_def'),E7p(top='E9',tag='E9_def'),KBp('E10',tag='E10_def'),
         KBp('E8',LCy=0.7,tag='E8_LCy0.7'),KBp('E10',LCy=0.7,tag='E10_LCy0.7')]
from topology.run_topology import run_topology
def fmt(v): return f'{float(v):.2f}'
def find_matrix(pt):
    toks=[f'{k}_{fmt(v)}' for k,v in pt['params'].items()]
    toks+= [f'x_{fmt(pt["x0"][0])}', f'y_{fmt(pt["x0"][1])}', f'z_{fmt(pt["x0"][2])}']
    for d in glob.glob(f"runs/{pt['topology']}_*l_max_{LMAX}"):
        if all(t in d for t in toks):
            f=os.path.join(d,f'TT_corr_matrix_l_2_{LMAX}_lp_2_{LMAX}.npy')
            if os.path.exists(f): return f
    return None
def ensure_matrix(pt):
    f=find_matrix(pt)
    if f: return f
    if left()<0: return None
    run_topology(topology=pt['topology'],l_max=LMAX,do_polarization=False,normalize=True,
                 l_range=np.array([[2,LMAX]]),lp_range=np.array([[2,LMAX]]),
                 x0=np.array(pt['x0']),**pt['params'])
    return find_matrix(pt)
print(len(POINTS),'点')

In [ ]:
# ---- 検証バッテリー（v0.3） ----
import camb as _camb
pars=_camb.CAMBparams(); pars.set_cosmology(H0=67.36,ombh2=0.02237,omch2=0.1200,tau=0.0544)
pars.InitPower.set_params(As=np.exp(3.044)*1e-10,ns=0.9649); pars.set_for_lmax(64,lens_potential_accuracy=1)
CL=_camb.get_results(pars).get_cmb_power_spectra(pars,CMB_unit='muK',raw_cl=True)['lensed_scalar'][:,0]
LM_LIST=[(l,m) for l in range(2,LMAX+1) for m in range(-l,l+1)]
def verify_matrix(M,tag):
    """エルミート性(必須) + 物理重みの健全性帯[0.5,2.0](規格化誤り検出) を検査し，
    位相による ℓ別パワー変調 rel_ℓ = (⟨対角⟩_ℓ/C_ℓ)/(⟨対角⟩_2/C_2) を情報として返す"""
    ok_h=np.allclose(M,M.conj().T,atol=1e-8*np.abs(M).max())
    diag=np.real(np.diag(M))
    per={l:np.mean([diag[i] for i,(li,_) in enumerate(LM_LIST) if li==l]) for l in range(2,LMAX+1)}
    rel={l:float((per[l]/CL[l])/(per[2]/CL[2])) for l in per}
    ok_w=all(0.5<r<2.0 for r in rel.values())
    if not ok_h: print(f'  [警告 {tag}] エルミート性 FAIL')
    if not ok_w: print(f'  [警告 {tag}] 物理重み帯域外（規格化誤りの疑い）rel={ {l:round(r,2) for l,r in rel.items()} }')
    return rel
ISO=np.diag(np.array([CL[l] for l,_ in LM_LIST],dtype=complex))
print('検証バッテリー準備OK（エルミート性＋健全性帯[0.5,2.0]・変調はCSVに記録）')

In [ ]:
# ---- 符号地図 v2（大域走査＋全不変軸の固定評価） ----
nside=8; mask=np.ones(hp.nside2npix(nside),bool)
ms=p2.MirrorStat(nside,mask); mb=pm.MirrorBatch(ms)
def axis_pix(n): 
    th=np.arccos(np.clip(n[2],-1,1)); ph=np.arctan2(n[1],n[0])%(2*np.pi)
    return int(hp.ang2pix(nside,th,ph))
def analyze(M,axdict,n=400):
    maps=tb.alms_to_maps(tb.sample_alms(M,LM_LIST,n,seed0=0),nside)
    Spg,Smg,_,_=mb.min_S_batch(maps,mondip=False,return_argmin=True)
    med=np.array([np.median(pm.scan_S(ms,maps[r],mondip=False)[0]) for r in range(80)])
    out=dict(minSp=float(np.median(Spg[:80]/med)),minSm=float(np.median(Smg[:80]/med)))
    for name,nvec in axdict.items():
        fa=pm.FixedAxisMirror(ms,axis_pix(nvec))
        Sp=np.empty(n);Sm=np.empty(n)
        for r in range(n):
            T=np.where(fa.mask,maps[r],0.0); s=0.5*(T+T[fa.r]); a=0.5*(T-T[fa.r])
            Sp[r]=np.sum(fa.v*s*s)/fa.cnt; Sm[r]=np.sum(fa.v*a*a)/fa.cnt
        key=axkey(name)
        out[f'Sp_{key}']=float(np.median(Sp[:80]/med)); out[f'Sm_{key}']=float(np.median(Sm[:80]/med))
    return out
rows=[]
iso_ref=analyze(ISO,{'refl:ŷ':np.array([0,1,0.])})
print('等方基準(CAMB重み):',{k:round(v,3) for k,v in iso_ref.items()})
for pt in POINTS:
    f=ensure_matrix(pt)
    if f is None: print(f"[{pt['tag']}] 未計算（時間予算）"); continue
    M=np.load(f)
    rel=verify_matrix(M,pt['tag'])
    r=analyze(M,AXES[pt['topology']]); r['tag']=pt['tag']
    for l,v in rel.items(): r[f'rel_l{l}']=round(v,3)
    rows.append(r)
    star=' ★S⁺抑制' if any(r[k]<r[k.replace('Sp_','Sm_')] for k in r if k.startswith('Sp_')) or r['minSp']<r['minSm'] else ''
    print(f"{pt['tag']:16s} " + ' '.join(f'{k}={v:.3f}' for k,v in r.items()
          if k!='tag' and not k.startswith('rel_')) + star)
pd.DataFrame(rows).to_csv(os.path.join(BASE,'t1_sign_map_v2.csv'),index=False)
print('→ t1_sign_map_v2.csv 保存')

In [ ]:
# ---- E7系のv0.1一致検証（新旧プログラムの相互検証） ----
REF={'E7_L1y1.0_g0':(1.013,0.671),'E7_L1y1.0_gq':(0.993,0.693),'E7_L1y1.0_gh':(1.019,0.691),
     'E7_L1y0.85_g0':(1.073,0.640),'E7_L1y0.85_gq':(1.077,0.633),'E7_L1y0.85_gh':(1.057,0.654),
     'E7_L1y0.7_g0':(1.043,0.672),'E7_L1y0.7_gq':(1.069,0.661),'E7_L1y0.7_gh':(1.090,0.630),
     'E7_L1y0.6_g0':(1.138,0.568),'E7_L1y0.6_gq':(1.117,0.600),'E7_L1y0.6_gh':(1.115,0.587),
     'E7_LAx0.7':(1.023,0.714),'E7_LAx1.3':(1.012,0.676),'E7_tilt':(0.996,0.708)}
df=pd.read_csv(os.path.join(BASE,'t1_sign_map_v2.csv'))
bad=0
for tag,(sp,sm) in REF.items():
    row=df[df.tag==tag]
    if len(row)==0: continue
    dsp=abs(float(row['Sp_refl_A'].iloc[0])-sp); dsm=abs(float(row['Sm_refl_A'].iloc[0])-sm)
    if dsp>0.02 or dsm>0.02: bad+=1; print(f'不一致 {tag}: Δ=({dsp:.3f},{dsm:.3f})')
print('E7一致検証:', 'PASS（全点±0.02以内）' if bad==0 else f'{bad}点不一致——報告してください')

In [ ]:
# ---- Tier B：等方極限の厳密検査（全辺1.4・±15%） ----
tb_fn=os.path.join(BASE,'t1_tierB_v2.json')
if not os.path.exists(tb_fn):
    ptB=dict(topology='E7',params=dict(LAx=1.4,LAy=1.4,L1y=1.4,L2x=1.4,L2z=1.4),
             x0=[0.,0.,0.],tag='TierB_iso_limit')
    f=ensure_matrix(ptB)
    M=np.load(f)
    diag=np.real(np.diag(M))
    per={l:np.mean([diag[i] for i,(li,_) in enumerate(LM_LIST) if li==l]) for l in range(2,LMAX+1)}
    rel={l:float((per[l]/CL[l])/(per[2]/CL[2])) for l in per}
    off=float(np.sum(np.abs(M-np.diag(np.diag(M)))**2)/np.sum(np.abs(M)**2))
    verdict='PASS' if all(0.85<r<1.15 for r in rel.values()) else 'FAIL'
    out=dict(rel={str(l):round(r,3) for l,r in rel.items()},offdiag_power_frac=round(off,4),verdict=verdict)
    json.dump(out,open(tb_fn,'w'))
    print('Tier B（等方極限）:',out)
else: print('Tier B: 済',json.load(open(tb_fn)))

## 完了後
`t1_sign_map_v2.csv`と両セルの画面出力（符号地図・E7一致検証）を添付してください。
E7一致がPASSなら新プログラムの正しさが旧結果で裏書きされ，E8/E10の
新規列（x̂面等）が族全体の符号問題への最終回答になります。